# Ablation 2026-09-01 — LDM + No Encoder + Temp only
**Run:** `abl_ldm_noenc_temp`  |  **W&B:** `1_Sep_2026_ldm_noenc_temp`  
**Model:** LDMModel (DDPM, unconditional)  |  **Fields:** `temperature` only


In [ ]:
RUN_NAME='abl_ldm_noenc_temp'; WANDB_RUN_NAME='1_Sep_2026_ldm_noenc_temp'
LDM_RUN_DIR='/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/ablation_20260901/abl_ldm_noenc_temp'
ENC_RUN_DIR=None
VAE_RUN_DIR='/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/ablation_20260901/vae_temp'
DATA_ROOT='/trace/group/forgelab/ngng/multifield/data_fields'
EVAL_OUT_DIR='/trace/group/forgelab/ngng/multifield/eval_results/ablation_20260901/abl_ldm_noenc_temp'
FIELD_NAMES=['temperature']; N_STEPS=3; DOWNSCALE_METHOD='direct'; NORMALIZE='standardize'
TIMESTEPS=1000; SCHEDULE='linear'; ENCODING=False; CONDITIONING='none'; DEVICE='cuda'
BATCH_INDEX=0; SAMPLE_INDEX=0; BATCH_SIZE=4; T_LIQ=1700.0; LIQ_THR=0.5
MELT_THRESHOLD=1900.0; ANALYSIS_CH=0; ANALYSIS_MAX_BATCH=None

See notebook 18 (`18_abl_ldm_noenc_sdf_results.ipynb`) for the full code — copy cells after this USER CONFIG. Only the paths and field config differ.

In [ ]:
%matplotlib inline
import os,sys,time; from pathlib import Path
import numpy as np,torch,pandas as pd; import matplotlib.pyplot as plt
from torch.utils.data import DataLoader; from scipy.ndimage import gaussian_filter as _gf
from IPython.display import display as _ipy_display
def _show(*a, **kw):
    for n in plt.get_fignums(): _ipy_display(plt.figure(n))
    plt.close('all')
plt.show = _show
if not torch.cuda.is_available() and DEVICE=='cuda': DEVICE='cpu'
def find_root(s=Path.cwd()):
    for p in [s,*s.parents]:
        if (p/'setup.py').exists() and (p/'diffusionsr').exists(): return p
    raise RuntimeError('no root')
PROJECT_ROOT=find_root()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0,str(PROJECT_ROOT))

In [ ]:
from diffusionsr.datasets.dataset import SimulationXZDataset
from diffusionsr.analysis.analysis_functions import get_profile
from diffusionsr.runners.train_ldm import LDMModel
def as_numpy(x): return x.detach().cpu().numpy() if isinstance(x,torch.Tensor) else np.asarray(x)
def mae_rmse(p,g): p,g=np.asarray(p).ravel(),np.asarray(g).ravel(); return {'MAE':float(np.mean(np.abs(p-g))),'RMSE':float(np.sqrt(np.mean((p-g)**2)))}
fn=FIELD_NAMES; has_sdf='sdfliqlabel' in fn; has_T='temperature' in fn; has_liq='liqlabel' in fn or has_sdf
kw=dict(downscale_method=DOWNSCALE_METHOD,root_folder=DATA_ROOT,normalize=NORMALIZE,n_steps=N_STEPS,field_names=FIELD_NAMES)
train_ds,dev_ds,test_ds=(SimulationXZDataset(split=s,**kw) for s in ['train','dev','test'])
model=LDMModel(vae_folder=VAE_RUN_DIR,results_folder=LDM_RUN_DIR,lr_encoder_folder=None,
    train_dataset=train_ds,dev_dataset=dev_ds,test_dataset=test_ds,
    timesteps=TIMESTEPS,conditioning=CONDITIONING,encoding=ENCODING,schedule=SCHEDULE,device=DEVICE,enc_output=False)
model.load_saved_model(); print(f'LDMModel (no encoder) loaded: {LDM_RUN_DIR}')

In [ ]:
# ── Melt-pool physical metrics engine ─────────────────────────────────────────
from diffusionsr.analysis.meltpool import MeltPoolMetrics

VOXEL_SIZE_UM   = (10.0, 10.0)   # canonical (depth, length) order; hrmesh=10 µm
PLATE_HEIGHT_UM = 400.0           # plate surface at depth pixel 40 × 10 µm

metrics_engine = MeltPoolMetrics.from_dataset(
    test_ds,
    spatial_axes=("length", "depth"),
    voxel_size_um=VOXEL_SIZE_UM,
    plate_height_um=PLATE_HEIGHT_UM,
)
_MP_METRICS_T = ['depth_um', 'length_um', 'area_um2']
_MP_METRICS_L = ['keyhole_depth_um', 'leading_wall_angle_deg'] if has_liq else []
print(f'MeltPoolMetrics ready.  available: {metrics_engine.available_metrics}')

In [ ]:
# ── Full test-set statistics — loaded from W&B artifact ───────────────────────
import os, wandb
WANDB_PROJECT = 'Flow3D_SuperResolution'
WANDB_ENTITY  = os.getenv('WANDB_ENTITY', '')

STATS_CH = (N_STEPS - 1) * len(FIELD_NAMES)

api = wandb.Api()
artifact = api.artifact(f'{WANDB_ENTITY}/{WANDB_PROJECT}/{RUN_NAME}_eval_predictions:latest')
artifact_dir = artifact.download()
data = np.load(os.path.join(artifact_dir, 'predictions.npz'), allow_pickle=True)
all_preds = data['pred']     # (N, C, H, W) physical units
all_gts   = data['gt']       # (N, C, H, W) physical units
all_vae   = data['vae_recon'] # (N, C, H, W) physical units
print(f'Loaded {len(all_preds)} test samples from W&B artifact')

maes, rmses, mp_errs, kh_errs, vc_maes, vae_maes = [], [], [], [], [], []
mp_depth_pred, mp_depth_gt   = [], []
mp_length_pred, mp_length_gt = [], []
mp_area_pred, mp_area_gt     = [], []
kh_depth_pred, kh_depth_gt   = [], []
lwa_pred, lwa_gt             = [], []

t0 = time.perf_counter()
for s in range(len(all_preds)):
    p  = all_preds[s]
    g  = all_gts[s]
    vp = all_vae[s]
    p_curr = p[STATS_CH:STATS_CH+len(FIELD_NAMES)]
    g_curr = g[STATS_CH:STATS_CH+len(FIELD_NAMES)]
    m = mae_rmse(p_curr[0], g_curr[0]); maes.append(m['MAE']); rmses.append(m['RMSE'])
    vae_maes.append(mae_rmse(vp[STATS_CH], g_curr[0])['MAE'])
    try:
        pmp, pkh = get_profile(p_curr[0:1]); gmp, gkh = get_profile(g_curr[0:1])
        mp_errs.append(float(np.mean(np.abs(pmp-gmp)))); kh_errs.append(float(np.mean(np.abs(pkh-gkh))))
    except: pass
    vc_maes.append(float(np.mean(np.abs((p_curr[0]>MELT_THRESHOLD).astype(float)-(g_curr[0]>MELT_THRESHOLD).astype(float)))))
    try:
        mp_r = metrics_engine(p, metrics=_MP_METRICS_T, basis='temperature')
        mp_g = metrics_engine(g, metrics=_MP_METRICS_T, basis='temperature')
        mp_depth_pred.append(float(mp_r['depth_um']));   mp_depth_gt.append(float(mp_g['depth_um']))
        mp_length_pred.append(float(mp_r['length_um'])); mp_length_gt.append(float(mp_g['length_um']))
        mp_area_pred.append(float(mp_r['area_um2']));    mp_area_gt.append(float(mp_g['area_um2']))
    except: pass
    if _MP_METRICS_L:
        try:
            kh_r = metrics_engine(p, metrics=_MP_METRICS_L, basis='liquid')
            kh_g = metrics_engine(g, metrics=_MP_METRICS_L, basis='liquid')
            kh_depth_pred.append(float(kh_r['keyhole_depth_um']))
            kh_depth_gt.append(float(kh_g['keyhole_depth_um']))
            if kh_r['leading_wall_angle_valid']: lwa_pred.append(float(kh_r['leading_wall_angle_deg']))
            if kh_g['leading_wall_angle_valid']: lwa_gt.append(float(kh_g['leading_wall_angle_deg']))
        except: pass

elapsed = time.perf_counter() - t0
print(f'n={len(maes)} DDPM (no enc)  MAE={np.nanmean(maes):.4f}±{np.nanstd(maes):.4f}')
print(f'  RMSE={np.nanmean(rmses):.4f}±{np.nanstd(rmses):.4f}')
print(f'  VAE recon MAE={np.nanmean(vae_maes):.4f}')
if mp_errs: print(f'  MP-MAE={np.nanmean(mp_errs):.2f}±{np.nanstd(mp_errs):.2f}px  KH-MAE={np.nanmean(kh_errs):.2f}px')
if vc_maes: print(f'  VC-MAE={np.nanmean(vc_maes):.4f}±{np.nanstd(vc_maes):.4f}')
if mp_depth_pred:
    print(f'  [phys] depth  pred={np.nanmean(mp_depth_pred):.1f}±{np.nanstd(mp_depth_pred):.1f}µm  gt={np.nanmean(mp_depth_gt):.1f}±{np.nanstd(mp_depth_gt):.1f}µm')
    print(f'  [phys] length pred={np.nanmean(mp_length_pred):.1f}±{np.nanstd(mp_length_pred):.1f}µm  gt={np.nanmean(mp_length_gt):.1f}±{np.nanstd(mp_length_gt):.1f}µm')
    print(f'  [phys] area   pred={np.nanmean(mp_area_pred):.0f}±{np.nanstd(mp_area_pred):.0f}µm²  gt={np.nanmean(mp_area_gt):.0f}±{np.nanstd(mp_area_gt):.0f}µm²')
if kh_depth_pred: print(f'  [phys] kh_depth pred={np.nanmean(kh_depth_pred):.1f}µm  gt={np.nanmean(kh_depth_gt):.1f}µm')
if lwa_pred: print(f'  [phys] lwa pred={np.nanmean(lwa_pred):.1f}°  gt={np.nanmean(lwa_gt):.1f}° (n={len(lwa_pred)})')
print(f'  Metric computation: {elapsed:.1f}s')
import os; os.makedirs(EVAL_OUT_DIR, exist_ok=True)
summary = {
    'run_name':RUN_NAME,'wandb_run':WANDB_RUN_NAME,'model':'LDM',
    'encoding':ENCODING,'conditioning':CONDITIONING,'fields':str(FIELD_NAMES),
    'sampler':'DDPM','n_test':len(maes),
    'mae_mean':float(np.nanmean(maes)),'mae_std':float(np.nanstd(maes)),
    'rmse_mean':float(np.nanmean(rmses)),'rmse_std':float(np.nanstd(rmses)),
    'vae_mae_mean':float(np.nanmean(vae_maes)) if vae_maes else float('nan'),
    'mp_mae_mean':float(np.nanmean(mp_errs)) if mp_errs else float('nan'),
    'kh_mae_mean':float(np.nanmean(kh_errs)) if kh_errs else float('nan'),
    'mp_depth_pred_mean':float(np.nanmean(mp_depth_pred)) if mp_depth_pred else float('nan'),
    'mp_depth_gt_mean':float(np.nanmean(mp_depth_gt)) if mp_depth_gt else float('nan'),
    'mp_length_pred_mean':float(np.nanmean(mp_length_pred)) if mp_length_pred else float('nan'),
    'mp_length_gt_mean':float(np.nanmean(mp_length_gt)) if mp_length_gt else float('nan'),
    'mp_area_pred_mean':float(np.nanmean(mp_area_pred)) if mp_area_pred else float('nan'),
    'mp_area_gt_mean':float(np.nanmean(mp_area_gt)) if mp_area_gt else float('nan'),
    'kh_depth_pred_mean':float(np.nanmean(kh_depth_pred)) if kh_depth_pred else float('nan'),
    'kh_depth_gt_mean':float(np.nanmean(kh_depth_gt)) if kh_depth_gt else float('nan'),
    'lwa_pred_mean':float(np.nanmean(lwa_pred)) if lwa_pred else float('nan'),
    'lwa_gt_mean':float(np.nanmean(lwa_gt)) if lwa_gt else float('nan'),
}
pd.DataFrame([summary]).to_csv(f'{EVAL_OUT_DIR}/metrics_summary.csv',index=False)
print(f'Saved -> {EVAL_OUT_DIR}/metrics_summary.csv')